# 07 Evaluation Comparison Results

Purpose: export the annotation file, compare methods, summarise coverage, and save final result tables and figures.

Inputs:
- `data/interim/jobs_clean.parquet`
- `data/interim/jobs_scope_audit.parquet`
- `data/processed/degree_profiles.parquet`
- `data/processed/jobs_with_clusters.parquet`
- `data/embeddings/similarity_matrix_all-MiniLM-L6-v2.npy`
- `data/embeddings/skill_coverage_matrix.npy`
- `data/embeddings/hybrid_matrix.npy`
- `data/embeddings/degree_cluster_similarity_k75.npy`

Outputs:
- `data/evaluation/golden_top10_jobs_per_degree.csv`
- `data/evaluation/method_metrics.csv`
- `data/evaluation/coverage_report.json`
- `data/processed/degree_job_alignment_summary.csv`
- `data/figures/degree_alignment_scores.png`
- `data/figures/alignment_heatmap.png`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.evaluation import build_coverage_report, build_degree_job_summary, evaluate_ranked_scores, export_golden_topk
from analysis.plotting import plot_alignment_heatmap, plot_degree_alignment_scores

MODEL_NAME = 'all-MiniLM-L6-v2'
N_CLUSTERS = 75

require_files([
    paths.jobs_clean_parquet,
    paths.jobs_scope_audit_parquet,
    paths.degree_profiles_parquet,
    paths.jobs_with_clusters_parquet,
    paths.similarity_matrix_npy(MODEL_NAME),
    paths.skill_coverage_matrix_npy,
    paths.hybrid_matrix_npy,
    paths.degree_cluster_similarity_npy(N_CLUSTERS),
    paths.cluster_summary_csv,
])
print('Expected upstream inputs: hybrid, clustering, and degree profile artifacts')


Expected upstream inputs: hybrid, clustering, and degree profile artifacts


In [2]:
jobs = pd.read_parquet(paths.jobs_clean_parquet)
jobs_scope_audit = pd.read_parquet(paths.jobs_scope_audit_parquet)
degree_profiles = pd.read_parquet(paths.degree_profiles_parquet)
jobs_with_clusters = pd.read_parquet(paths.jobs_with_clusters_parquet)
cluster_df = pd.read_csv(paths.cluster_summary_csv)
sim_matrix = np.load(paths.similarity_matrix_npy(MODEL_NAME))
skill_coverage_matrix = np.load(paths.skill_coverage_matrix_npy)
hybrid_matrix = np.load(paths.hybrid_matrix_npy)
degree_cluster_sim = np.load(paths.degree_cluster_similarity_npy(N_CLUSTERS))

summary_df = build_degree_job_summary(
    degree_profiles,
    jobs,
    hybrid_matrix,
    paths.degree_job_alignment_summary_csv,
    sim_matrix=sim_matrix,
    skill_coverage_matrix=skill_coverage_matrix,
    top_n=5,
)

expected_rows = len(degree_profiles) * 10
regenerate_golden = True
if paths.golden_top10_csv.exists():
    golden_top10_df = pd.read_csv(paths.golden_top10_csv)
    required_columns = {'degree_id', 'job_id', 'cosine_sim', 'skill_coverage', 'hybrid_score', 'human_label'}
    if required_columns.issubset(golden_top10_df.columns) and len(golden_top10_df) == expected_rows:
        regenerate_golden = False
        print(f'Loaded existing annotation file from: {paths.golden_top10_csv}')

if regenerate_golden:
    golden_top10_df = export_golden_topk(
        degree_profiles,
        jobs_with_clusters,
        sim_matrix,
        skill_coverage_matrix,
        hybrid_matrix,
        paths.golden_top10_csv,
        top_k=10,
        degree_cluster_sim=degree_cluster_sim,
    )
    print(f'Exported annotation file to: {paths.golden_top10_csv}')

metrics_df, correlations_df = evaluate_ranked_scores(
    golden_top10_df,
    score_columns=['cosine_sim', 'skill_coverage', 'hybrid_score', 'cluster_score'],
    metrics_path=paths.evaluation_metrics_csv,
)

method_summary = pd.DataFrame({
    'method': ['baseline_cosine', 'skill_coverage', 'hybrid', 'cluster'],
    'mean_top1_score': [
        float(sim_matrix.max(axis=1).mean()),
        float(skill_coverage_matrix.max(axis=1).mean()),
        float(hybrid_matrix.max(axis=1).mean()),
        float(degree_cluster_sim.max(axis=1).mean()),
    ],
    'mean_top5_score': [
        float(np.sort(sim_matrix, axis=1)[:, -5:].mean()),
        float(np.sort(skill_coverage_matrix, axis=1)[:, -5:].mean()),
        float(np.sort(hybrid_matrix, axis=1)[:, -5:].mean()),
        float(np.sort(degree_cluster_sim, axis=1)[:, -5:].mean()),
    ],
}).round(4)

plot_degree_alignment_scores(hybrid_matrix, degree_profiles, paths.degree_alignment_scores_png)
plot_alignment_heatmap(degree_cluster_sim, degree_profiles, cluster_df, paths.alignment_heatmap_png)

coverage_payload = {
    'jobs_final': int(len(jobs)),
    'jobs_scope_audit': int(len(jobs_scope_audit)),
    'degrees_profiled': int(len(degree_profiles)),
    'degree_job_pairs_scored': int(len(degree_profiles) * len(jobs)),
    'golden_rows': int(len(golden_top10_df)),
    'golden_labelled_rows': int(golden_top10_df['human_label'].apply(lambda value: str(value).strip()).isin(['0', '1', '2']).sum()),
    'summary_csv': str(paths.degree_job_alignment_summary_csv),
    'metrics_csv': str(paths.evaluation_metrics_csv),
    'golden_csv': str(paths.golden_top10_csv),
    'alignment_heatmap': str(paths.alignment_heatmap_png),
    'degree_scores_figure': str(paths.degree_alignment_scores_png),
}
build_coverage_report(coverage_payload, paths.coverage_report_json)


Loaded existing annotation file from: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/evaluation/golden_top10_jobs_per_degree.csv


PosixPath('/Users/marcusyeo/Github/DSA4264-Text-Group-3/data/evaluation/coverage_report.json')

In [3]:
print('Top-5 jobs per degree summary:')
display(summary_df.head(20))

print('Method comparison summary:')
display(method_summary)

if metrics_df.empty:
    print('No human labels found yet in the golden set. Fill the human_label column with 2, 1, or 0, then rerun this notebook to compute evaluation metrics.')
else:
    print('Per-degree evaluation metrics:')
    display(metrics_df.head(20))
    print('Method-level score-to-label correlations:')
    display(correlations_df)

print('Key outputs:')
for path in [
    paths.golden_top10_csv,
    paths.evaluation_metrics_csv,
    paths.coverage_report_json,
    paths.degree_job_alignment_summary_csv,
    paths.degree_alignment_scores_png,
    paths.alignment_heatmap_png,
]:
    print(f'- {path}')


Top-5 jobs per degree summary:


,degree_id,degree_name,rank,job_id,job_title,company,job_categories,hybrid_score,cosine_sim,skill_coverage
0,biz,Business Administration,1,MCF-2026-0172321,Financial Accountant,LICO RESOURCES PTE. LTD.,Accounting / Auditing / Taxation,0.3984,0.5691,0.0
1,biz,Business Administration,2,MCF-2026-0024504,Finance Controller (Construction & Manufacturing),KINGSFORCE MANAGEMENT SERVICES PTE LTD,"Accounting / Auditing / Taxation, Banking and ...",0.3827,0.5467,0.0
2,biz,Business Administration,3,MCF-2025-1560828,Accountant,"ALTRAD FRP PRODUCTS CO., PTE. LTD.",Accounting / Auditing / Taxation,0.3823,0.5462,0.0
3,biz,Business Administration,4,MCF-2026-0169544,Finance Executive,CECE CONSULT PTE. LTD.,Accounting / Auditing / Taxation,0.3819,0.5456,0.0
4,biz,Business Administration,5,MCF-2026-0189966,Senior Accountant/Finance Manager,RADUGA PTE LTD,Accounting / Auditing / Taxation,0.3774,0.5392,0.0
5,ce,Civil Engineering,1,MCF-2026-0178508,Design Engineer/Senior Design Engineer,SAMWOH CORPORATION PTE. LTD.,"Building and Construction, Engineering",0.3804,0.5434,0.0
6,ce,Civil Engineering,2,MCF-2026-0192420,Civil & Structural Engineer / Assistant Civil ...,BUILDBRIDGE PARTNERS PTE. LTD.,"Building and Construction, Consulting, Enginee...",0.3579,0.5113,0.0
7,ce,Civil Engineering,3,MCF-2026-0180989,Project Engineer _Civil & Structural,SUN DEMOLITION PTE. LTD.,Building and Construction,0.3572,0.5103,0.0
8,ce,Civil Engineering,4,MCF-2026-0125621,"Civil & Structural Engineer (Main Con, Tender)",RECRUIT EXPERT PTE. LTD.,Building and Construction,0.3562,0.5089,0.0
9,ce,Civil Engineering,5,MCF-2026-0176736,"Civil, Structural and Geotechnical Design Engi...",VENGINEERS CONSULTANCY PTE. LTD.,"Building and Construction, Consulting, Enginee...",0.3553,0.5075,0.0


Method comparison summary:


,method,mean_top1_score,mean_top5_score
0,baseline_cosine,0.5426,0.5235
1,skill_coverage,0.0000,0.0000
2,hybrid,0.3798,0.3664
3,cluster,0.5176,0.4693


No human labels found yet in the golden set. Fill the human_label column with 2, 1, or 0, then rerun this notebook to compute evaluation metrics.
Key outputs:
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/evaluation/golden_top10_jobs_per_degree.csv
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/evaluation/method_metrics.csv
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/evaluation/coverage_report.json
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/degree_job_alignment_summary.csv
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/figures/degree_alignment_scores.png
- /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/figures/alignment_heatmap.png
